# Phase 1 — INT8 Baseline Reproduction

**Project**: Quantum-Inspired Tensor-Network Compression of OpenVLA-7B  
**Challenge**: Global Quantum + AI Challenge 2026 — VW Group Enterprise Track  
**Phase**: 1 of 7 | **Execution**: Google Colab (GPU runtime required)

---

## What this notebook does

Loads OpenVLA-7B in INT8 via bitsandbytes (the accepted baseline per challenge §5.4),
runs inference on 200 held-out Open X-Embodiment `bridge_dataset` episodes for each
of the three evaluation seeds, and writes `results/baseline_metrics.json` with all
§5.5 benchmark fields.

**Metrics recorded** (mean ± std, 3 independent runs):
- Action-prediction L1 error
- Per-sample inference time (ms)
- Peak GPU memory (MiB)
- GPU power draw (W) and total energy (kWh)
- Total parameter count

## Before running

1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU or A100).
2. Add two secrets in Colab Secrets (left panel → key icon):
   - `HF_TOKEN` — HuggingFace token with read access to `openvla/openvla-7b`
     (accept the [LLaMA-2 Community License](https://huggingface.co/meta-llama/Llama-2-7b)
     and the [OpenVLA model agreement](https://huggingface.co/openvla/openvla-7b) first)
   - `GH_TOKEN` — GitHub Personal Access Token with `repo` scope
     (Settings → Developer settings → Personal access tokens → Tokens (classic))
3. **Run cells in order from the top.** Do not skip or reorder.

---

> **License notice** — OpenVLA-7B *code* is MIT.
> The *model weights* are subject to the **LLaMA-2 Community License**
> (non-commercial research use only). Downloading the weights via HuggingFace Hub
> constitutes acceptance of that license.

In [ ]:
# ── Cell 1: Install / pin dependencies ───────────────────────────────────────
# Uses subprocess so pip runs in the current Python env before any imports.
#
# OpenVLA version pins (from openvla/openvla pyproject.toml):
#   transformers==4.40.1  — only version tested by OpenVLA authors
#   tokenizers==0.19.1   — must match transformers 4.40.1
#   timm==0.9.10         — imported by modeling_prismatic.py via trust_remote_code
#   sentencepiece==0.1.99 — required by the LLaMA-2 tokenizer
#
# Data loading uses HuggingFace Hub datasets + PyAV (no tensorflow-datasets).
# tfds was dropped: Colab's pre-installed TF ships protobuf 5.x at runtime
# but the current tfds wheel embeds protobuf 6.x gencode, making the module
# unimportable with "Detected incompatible Protobuf Gencode/Runtime versions".
import subprocess, sys

def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

_pip(
    "transformers==4.40.1",
    "tokenizers==0.19.1",
    "timm==0.9.10",
    "sentencepiece==0.1.99",
)
_pip(
    "accelerate>=0.27.0",
    "bitsandbytes>=0.43.0",
    "pynvml",
    "av",       # PyAV — decode MP4 video frames from HF LeRobot dataset
    "datasets", # HuggingFace datasets library for streaming bridge data
    "PyYAML",
    "tqdm",
)
print("pip installs complete.")

In [ ]:
# ── Cell 2: Verify transformers version and key imports ──────────────────────
# Fails loudly before the 15 GB model download if the wrong version landed.
# If this cell raises, do: Runtime → Disconnect and delete runtime, then
# re-run from Cell 1 without running anything else first.
import transformers

REQUIRED_TRANSFORMERS = "4.40.1"
_ver = transformers.__version__
print(f"transformers : {_ver}")

if _ver != REQUIRED_TRANSFORMERS:
    raise RuntimeError(
        f"transformers {_ver} is installed but OpenVLA requires exactly {REQUIRED_TRANSFORMERS}.\n"
        f"Other 4.x builds and all 5.x builds break modeling_prismatic.py\n"
        f"(_supports_sdpa missing, is_flax_available moved, etc.).\n\n"
        f"Fix:\n"
        f"  1. Runtime → Disconnect and delete runtime\n"
        f"  2. Re-open the notebook and run Cell 1 before anything else\n"
        f"  3. Do NOT run any other cells before Cell 1 finishes"
    )
print(f"  ✓  {REQUIRED_TRANSFORMERS} confirmed")

try:
    from transformers import AutoModelForVision2Seq, AutoProcessor
    print("AutoModelForVision2Seq : importable ✓")
    print("AutoProcessor          : importable ✓")
except ImportError as exc:
    raise ImportError(
        f"transformers {_ver} installed but import failed: {exc}\n"
        "Check pip output in Cell 1 for conflicts."
    )

In [ ]:
# ── Cell 3: Clone repo and install the vlam_compress package ─────────────────
import os, sys, subprocess

# ── Environment detection ─────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists("/kaggle/working")
IS_COLAB  = False
try:
    import google.colab; IS_COLAB = True
except ImportError:
    pass

REPO_DIR = "/kaggle/working/vlam" if IS_KAGGLE else "/content/vlam"

# ── Secret fetch ──────────────────────────────────────────────────────────────
_gh_token = ""
if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        _gh_token = UserSecretsClient().get_secret("GH_TOKEN")
        print("GH_TOKEN loaded from Kaggle Secrets.")
    except Exception as _e:
        print(f"GH_TOKEN not found in Kaggle Secrets ({_e}).")
elif IS_COLAB:
    try:
        from google.colab import userdata
        _gh_token = userdata.get("GH_TOKEN")
        print("GH_TOKEN loaded from Colab Secrets.")
    except Exception as _e:
        print(f"GH_TOKEN not found in Colab Secrets ({_e}).")
else:
    _gh_token = os.environ.get("GH_TOKEN", "")

if _gh_token:
    REPO_URL = f"https://{_gh_token}@github.com/quantumadventurer11/vw-quantum-vlam-challenge"
else:
    print("No GH_TOKEN available — falling back to unauthenticated URL.")
    REPO_URL = "https://github.com/quantumadventurer11/vw-quantum-vlam-challenge"

# ── Clone / pull ──────────────────────────────────────────────────────────────
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print(f"Working directory : {os.getcwd()}")
print(f"Environment       : {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'local'}")
print("vlam_compress package installed.")

In [ ]:
# ── Cell 4: HuggingFace authentication ───────────────────────────────────────
# Requires an HF_TOKEN secret (read access to openvla/openvla-7b, gated repo).
# Colab: add in Colab Secrets (left panel → key icon)
# Kaggle: add in Kernel Settings → Secrets → Add New Secret
from huggingface_hub import login

_hf_token = ""
if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        _hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded from Kaggle Secrets.")
    except Exception as _e:
        print(f"HF_TOKEN not found in Kaggle Secrets ({_e}).")
elif IS_COLAB:
    try:
        from google.colab import userdata
        _hf_token = userdata.get("HF_TOKEN")
        print("HF_TOKEN loaded from Colab Secrets.")
    except Exception as _e:
        print(f"HF_TOKEN not found in Colab Secrets ({_e}).")
else:
    import os; _hf_token = os.environ.get("HF_TOKEN", "")

if _hf_token:
    login(token=_hf_token, add_to_git_credential=False)
else:
    print("No HF_TOKEN found. Attempting interactive login...")
    login()

In [ ]:
# ── Cell 4: Google Cloud auth — not needed (HF Hub replaces GCS) ─────────────
# The dataset was previously loaded from gs://gresearch/robotics/bridge_dataset
# via tensorflow-datasets. We now stream from HuggingFace Hub (lerobot/bridge),
# which requires no Google Cloud credentials.
print("GCS auth not required — data loads from HuggingFace Hub.")

In [ ]:
# ── Cell 5: Imports ───────────────────────────────────────────────────────────
import json
import platform
import time
from pathlib import Path

import numpy as np
import pynvml
import torch
import transformers
import yaml
from datasets import load_dataset
from PIL import Image
from tqdm.auto import tqdm

import bitsandbytes as bnb  # noqa: F401  (confirms install)

print("All imports OK.")

In [ ]:
# ── Cell 6: Environment diagnostics (logged to results for §6 Resource Declaration)
import bitsandbytes

assert torch.cuda.is_available(), "No GPU found — change runtime to GPU and restart."

props = torch.cuda.get_device_properties(0)
print(f"PyTorch          : {torch.__version__}")
print(f"CUDA             : {torch.version.cuda}")
print(f"Transformers     : {transformers.__version__}")
print(f"bitsandbytes     : {bitsandbytes.__version__}")
print(f"GPU name         : {props.name}")
print(f"GPU VRAM total   : {props.total_memory / 1024**3:.1f} GB")
print(f"GPU multi-proc   : {props.multi_processor_count}")
print(f"Platform         : {platform.platform()}")

In [ ]:
# ── Cell 7: Load project config ───────────────────────────────────────────────
with open("configs/seeds.yaml") as f:
    cfg = yaml.safe_load(f)

SEEDS           = cfg["seeds"]               # [42, 1337, 2024]
N_EVAL_EPISODES = cfg["eval"]["n_episodes"]  # 200
MODEL_ID        = "openvla/openvla-7b"
UNNORM_KEY      = "bridge_orig"              # BridgeV2 action normalization stats
HF_DATASET_ID   = "lerobot/bridge"          # Bridge V2 on HuggingFace Hub

if IS_KAGGLE:
    RESULTS_DIR = Path("/kaggle/working/results")
else:
    RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Seeds            : {SEEDS}")
print(f"Episodes per run : {N_EVAL_EPISODES}")
print(f"Model            : {MODEL_ID}")
print(f"Dataset          : {HF_DATASET_ID}")
print(f"Results dir      : {RESULTS_DIR}")

In [ ]:
# ── Cell 8: Initialize pynvml (GPU power monitoring) ─────────────────────────
pynvml.nvmlInit()
gpu_handle = pynvml.nvmlDeviceGetHandleByIndex(0)

driver_ver = pynvml.nvmlSystemGetDriverVersion()
if isinstance(driver_ver, bytes):
    driver_ver = driver_ver.decode()

print(f"GPU name (nvml)  : {pynvml.nvmlDeviceGetName(gpu_handle)}")
print(f"Driver version   : {driver_ver}")
print(f"Current power    : {pynvml.nvmlDeviceGetPowerUsage(gpu_handle) / 1000:.1f} W")

In [ ]:
# ── Cell 9: Load OpenVLA-7B ───────────────────────────────────────────────────
# LLaMA-2 Community License — non-commercial research only.
# attn_implementation='eager': OpenVLA's modeling_prismatic.py lacks _supports_sdpa.
#
# bitsandbytes INT8 requires sm_75+ (Turing: T4/A100/H100).
# GPUs below sm_75 (P100=sm_60, V100=sm_70) fall back to FP16 (~14 GB VRAM).
#
# INT8 monkey-patch: accelerate.dispatch_model calls model.to(device) after
# bitsandbytes quantization; transformers 4.40.1 raises ValueError for that.
# Patch makes .to() a no-op for already-quantized models; restored via finally.
import transformers
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig

sm_major, sm_minor = torch.cuda.get_device_capability(0)
USE_INT8 = (sm_major * 10 + sm_minor) >= 75
print(f"GPU capability : sm_{sm_major}{sm_minor}  →  {'INT8 quantization' if USE_INT8 else 'FP16 (no bitsandbytes INT8 on sm_60/70)'}")

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
torch.cuda.reset_peak_memory_stats()

if USE_INT8:
    print("Loading model in INT8 (bitsandbytes, sm_75+)...")
    _orig_to = transformers.modeling_utils.PreTrainedModel.to
    def _bnb_safe_to(self, *args, **kwargs):
        if getattr(self, "is_loaded_in_8bit", False):
            return self  # already on CUDA; skip guard
        return _orig_to(self, *args, **kwargs)
    transformers.modeling_utils.PreTrainedModel.to = _bnb_safe_to
    try:
        bnb_config = BitsAndBytesConfig(
            load_in_8bit=True,
            llm_int8_enable_fp32_cpu_offload=True,
        )
        model = AutoModelForVision2Seq.from_pretrained(
            MODEL_ID,
            attn_implementation="eager",
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
    finally:
        transformers.modeling_utils.PreTrainedModel.to = _orig_to
else:
    # P100 / V100: FP16, ~14 GB VRAM.  device_map="auto" splits to CPU if needed.
    print("Loading model in FP16 (~14 GB VRAM)...")
    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        attn_implementation="eager",
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )

model.eval()
peak_mem_load_mib = torch.cuda.max_memory_allocated() / (1024 ** 2)
print(f"Model loaded. Precision: {'INT8' if USE_INT8 else 'FP16'}. "
      f"Peak GPU mem: {peak_mem_load_mib:.0f} MiB")

In [ ]:
# ── Cell 10: Model diagnostics — validate auto class, predict_action, unnorm_key
# Run this immediately after loading; it raises informative errors rather than
# letting KeyError or AttributeError surface inside the evaluation loop later.

print("── Model diagnostics ─────────────────────────────────────────────────────")

# 1. Confirm the resolved class looks like an OpenVLA model.
model_class = type(model).__name__
print(f"Loaded class    : {model_class}")
_vla_keywords = ("openvla", "action", "vla")
if any(kw in model_class.lower() for kw in _vla_keywords):
    print("  ✓  Class name contains an OpenVLA keyword — looks correct.")
else:
    print(f"  ⚠  WARNING: '{model_class}' doesn't look like an OpenVLA class.")
    print("     Expected something like 'OpenVLAForActionPrediction'.")
    print("     If predict_action is missing below, try:")
    print("       from transformers import AutoModel")
    print("       model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True, ...)")

# 2. Confirm predict_action is available — fail loudly before the eval loop.
if not (hasattr(model, "predict_action") and callable(model.predict_action)):
    raise RuntimeError(
        f"model.predict_action is not available (model class: {model_class}).\n"
        "The model loaded but does not expose the OpenVLA action-prediction API.\n"
        "Possible causes:\n"
        "  • trust_remote_code=True was not applied\n"
        "  • AutoModelForVision2Seq resolved to the wrong class\n"
        "Fix: try loading with AutoModel instead of AutoModelForVision2Seq."
    )
print("predict_action  : available ✓")

# 3. Validate UNNORM_KEY against model.config.norm_stats.
if not (hasattr(model.config, "norm_stats") and model.config.norm_stats):
    print(f"\n  ⚠  model.config.norm_stats not found or empty.")
    print(f"     predict_action will likely fail on unnorm_key='{UNNORM_KEY}'.")
    print("     Inspect model.config directly: print(vars(model.config))")
else:
    available_keys = list(model.config.norm_stats.keys())
    print(f"\nAvailable unnorm keys ({len(available_keys)} total):")
    for k in available_keys:
        tag = "  ← UNNORM_KEY" if k == UNNORM_KEY else ""
        print(f"  '{k}'{tag}")

    if UNNORM_KEY not in available_keys:
        bridge_candidates = [k for k in available_keys if "bridge" in k.lower()]
        hint = (
            f"Likely bridge key(s): {bridge_candidates}"
            if bridge_candidates
            else f"No 'bridge' key found. Full list: {available_keys}"
        )
        raise KeyError(
            f"\n\nUNNORM_KEY='{UNNORM_KEY}' is not in model.config.norm_stats.\n"
            f"{hint}\n\n"
            f"Fix: update UNNORM_KEY in the config cell to one of the listed values "
            f"and re-run from that cell."
        )
    print(f"\n  ✓  UNNORM_KEY='{UNNORM_KEY}' confirmed present in norm_stats.")

print("──────────────────────────────────────────────────────────────────────────")

In [ ]:
# ── Cell 10: Parameter count and hardware profile ─────────────────────────────
import bitsandbytes as bnb_module

n_params = sum(p.numel() for p in model.parameters())
n_params_int8 = sum(
    p.numel() for p in model.parameters()
    if hasattr(p, 'quant_state') or p.dtype == torch.int8
)

print(f"Total parameters : {n_params:,} ({n_params / 1e9:.3f}B)")
print(f"INT8 parameters  : {n_params_int8:,} ({n_params_int8 / n_params * 100:.1f}%)")
print(f"Peak mem (load)  : {peak_mem_load_mib:.0f} MiB")

# Build hardware profile — saved with results for §6 Resource Declaration
props = torch.cuda.get_device_properties(0)
hardware_profile = {
    "gpu_name": props.name,
    "gpu_vram_gb": round(props.total_memory / 1024 ** 3, 2),
    "gpu_multi_proc": props.multi_processor_count,
    "cuda_version": torch.version.cuda,
    "driver_version": driver_ver,
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "bitsandbytes_version": bnb_module.__version__,
    "platform": platform.platform(),
    "n_params_total": n_params,
    "n_params_b": round(n_params / 1e9, 3),
    "peak_mem_load_mib": round(peak_mem_load_mib, 1),
    "model_precision": "INT8" if USE_INT8 else "FP16",
    "sm_version": f"sm_{sm_major}{sm_minor}",
}

# Persist hardware profile for Phase 7 and the report
hw_path = Path("configs/hardware_profile.yaml")
with open(hw_path, "w") as f:
    yaml.dump(hardware_profile, f, default_flow_style=False)
print(f"Hardware profile written to {hw_path}")

In [ ]:
# ── Cell 11: Dataset helpers ──────────────────────────────────────────────────
# task_map is populated by Cell 12 after loading the dataset metadata.
# Initialise to empty here so this cell can be re-run safely without Cell 12.
task_map: dict = {}


def hf_item_to_sample(item: dict) -> dict:
    """
    Convert a HuggingFace (LeRobot) dataset item to a unified dict:
      image     : PIL.Image.Image
      language  : str
      action_gt : np.ndarray  shape (7,)

    Handles LeRobot v1 (image bytes in parquet) and v2 (video-encoded frames).
    """
    # ── Image ─────────────────────────────────────────────────────────────────
    img = None
    for _key in (
        "observation.image",
        "observation.images.image_0",
        "observation.images.image",
        "image",
        "rgb",
    ):
        if _key in item:
            img = item[_key]
            break
    if img is None:
        for _key in item:
            if "image" in _key.lower():
                img = item[_key]
                break
    if img is None:
        raise KeyError(f"No image key found. Available: {list(item.keys())}")

    if isinstance(img, dict):
        _path = img.get("path")
        _bytes = img.get("bytes")
        if _path is not None:
            # LeRobot v2: video-encoded — seek to timestamp with PyAV
            import av
            _ts = float(img.get("timestamp") or 0.0)
            with av.open(_path) as _c:
                _stream = _c.streams.video[0]
                if _ts > 0:
                    _pts = int(_ts / float(_stream.time_base))
                    _c.seek(_pts, stream=_stream)
                for _f in _c.decode(video=0):
                    img = Image.fromarray(_f.to_ndarray(format="rgb24"))
                    break
        elif _bytes is not None:
            # LeRobot v1: image bytes embedded in parquet
            import io
            img = Image.open(io.BytesIO(_bytes))
        else:
            raise TypeError(f"Unrecognised image dict keys: {list(img.keys())}")
    elif isinstance(img, np.ndarray):
        img = Image.fromarray(img.astype(np.uint8))

    if not isinstance(img, Image.Image):
        raise TypeError(f"Could not decode image (got {type(img).__name__})")

    # ── Action ────────────────────────────────────────────────────────────────
    action = item.get("action")
    if action is None:
        raise KeyError(f"'action' not in item. Available: {list(item.keys())}")
    if isinstance(action, dict):
        _parts = [np.array(v, dtype=np.float32).ravel() for v in action.values()]
        action_gt = np.concatenate(_parts)[:7]
    else:
        action_gt = np.array(action, dtype=np.float32).ravel()[:7]

    # ── Language instruction ──────────────────────────────────────────────────
    lang = (
        item.get("language_instruction")
        or item.get("task")
        or item.get("instruction")
    )
    if lang is None:
        _tidx = item.get("task_index")
        lang = task_map.get(_tidx, "") if _tidx is not None else ""
    if isinstance(lang, (bytes, bytearray)):
        lang = lang.decode("utf-8")
    lang = str(lang).strip() or "pick up the object"

    return {"image": img, "language": lang, "action_gt": action_gt}


print("Dataset helper defined.")

In [ ]:
# ── Cell 12: Load bridge_dataset from HuggingFace Hub ────────────────────────
# Uses the lerobot/bridge dataset mirror (LeRobot v2.0 format).
# Images are stored as video-encoded MP4 chunks; PyAV decodes frames on demand.
# No Google Cloud credentials needed — data streams from HuggingFace Hub.
import json as _json
from huggingface_hub import hf_hub_download

print(f"Loading {HF_DATASET_ID} from HuggingFace Hub (streaming mode)...")
hf_ds = load_dataset(HF_DATASET_ID, split="train", streaming=True)

# Load task_index → language instruction mapping from dataset metadata
try:
    _tasks_file = hf_hub_download(HF_DATASET_ID, filename="meta/tasks.jsonl")
    with open(_tasks_file) as _f:
        for _line in _f:
            _d = _json.loads(_line.strip())
            task_map[_d["task_index"]] = _d["task"]
    print(f"Loaded {len(task_map)} tasks from meta/tasks.jsonl")
except Exception as _e:
    print(f"Could not load tasks.jsonl ({_e}) — generic language fallback active.")

# Schema peek: consume one item, inspect columns, create fresh iterator
_peek = next(iter(hf_ds))
print(f"\nDataset columns ({len(_peek)}):")
for _col, _val in _peek.items():
    _dtype = type(_val).__name__
    if isinstance(_val, (list, tuple)):
        _dtype = f"list[{len(_val)}]"
    elif isinstance(_val, dict):
        _dtype = f"dict{{{', '.join(_val.keys())}}}"
    print(f"  {_col}: {_dtype}")
del _peek

print("\nDataset ready — frames streamed on demand per seed.")

In [ ]:
# ── Cell 13: Warm-up inference pass ──────────────────────────────────────────
# Prime the GPU kernels so the first timed run is not penalised by JIT compilation.

print("Running warm-up inference (3 forward passes)...")

warmup_sample = hf_item_to_sample(next(iter(hf_ds)))

warmup_inputs = processor(
    warmup_sample["language"], warmup_sample["image"]
).to("cuda:0")

with torch.no_grad():
    for _ in range(3):
        _ = model.predict_action(**warmup_inputs, unnorm_key=UNNORM_KEY, do_sample=False)
torch.cuda.synchronize()

torch.cuda.reset_peak_memory_stats()  # reset after warm-up so baseline is clean
print("Warm-up complete. Peak memory stats reset.")

In [ ]:
# ── Cell 14: Timed inference helper ──────────────────────────────────────────

def run_inference_timed(
    image: Image.Image,
    language: str,
    handle,
) -> tuple:
    """
    Run one forward pass and return:
      (action_pred, elapsed_ms, peak_mem_mib, avg_power_w)

    Power is sampled immediately before and after the forward pass;
    the average is used as the representative power for that sample.
    """
    torch.cuda.reset_peak_memory_stats()

    inputs = processor(language, image).to("cuda:0")

    pwr_before_w = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    with torch.no_grad():
        action_pred = model.predict_action(
            **inputs, unnorm_key=UNNORM_KEY, do_sample=False
        )

    torch.cuda.synchronize()
    t1 = time.perf_counter()

    pwr_after_w = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0

    elapsed_ms   = (t1 - t0) * 1000.0
    peak_mem_mib = torch.cuda.max_memory_allocated() / (1024 ** 2)
    avg_power_w  = (pwr_before_w + pwr_after_w) / 2.0

    return action_pred, elapsed_ms, peak_mem_mib, avg_power_w


print("Inference helper defined.")

In [ ]:
# ── Cell 15: Main evaluation loop (3 seeds × 200 frames) ─────────────────────
# Each seed shuffles the dataset stream differently → independent frame subsets.
# PyTorch RNG is also seeded for full reproducibility.

results_per_seed = {}

for seed in SEEDS:
    print(f"\n{'─' * 60}")
    print(f"Seed {seed}  |  evaluating {N_EVAL_EPISODES} frames")
    print(f"{'─' * 60}")

    torch.manual_seed(seed)
    np.random.seed(seed)

    ds_eval = (
        hf_ds
        .shuffle(seed=seed, buffer_size=5000)
        .take(N_EVAL_EPISODES)
    )

    l1_errors        = []
    times_ms         = []
    peak_mems_mib    = []
    power_readings_w = []
    skipped          = 0

    wall_t0 = time.perf_counter()

    for item_idx, item in enumerate(
        tqdm(ds_eval, total=N_EVAL_EPISODES, desc=f"seed={seed}")
    ):
        try:
            sample = hf_item_to_sample(item)
        except (KeyError, TypeError, ValueError) as exc:
            print(f"  item {item_idx}: skipped ({exc})")
            skipped += 1
            continue

        action_pred, t_ms, mem_mib, pwr_w = run_inference_timed(
            sample["image"], sample["language"], gpu_handle
        )

        action_gt = sample["action_gt"]
        min_len = min(len(action_pred), len(action_gt))
        l1 = float(np.abs(action_pred[:min_len] - action_gt[:min_len]).mean())

        l1_errors.append(l1)
        times_ms.append(t_ms)
        peak_mems_mib.append(mem_mib)
        power_readings_w.append(pwr_w)

    wall_elapsed_s = time.perf_counter() - wall_t0
    avg_pwr_w      = float(np.mean(power_readings_w)) if power_readings_w else 0.0
    total_kwh      = (avg_pwr_w * wall_elapsed_s) / 3.6e6
    co2e_g         = total_kwh * 386.0  # US average: 0.386 kg CO2e/kWh

    seed_result = {
        "seed": seed,
        "n_samples": len(l1_errors),
        "n_skipped": skipped,
        "l1_error_mean": float(np.mean(l1_errors)),
        "l1_error_std":  float(np.std(l1_errors)),
        "inference_time_ms_mean": float(np.mean(times_ms)),
        "inference_time_ms_std":  float(np.std(times_ms)),
        "peak_mem_mib_mean": float(np.mean(peak_mems_mib)),
        "peak_mem_mib_std":  float(np.std(peak_mems_mib)),
        "avg_power_w":  avg_pwr_w,
        "wall_time_s":  round(wall_elapsed_s, 2),
        "total_kwh":    round(total_kwh, 6),
        "co2e_g":       round(co2e_g, 4),
    }
    results_per_seed[str(seed)] = seed_result

    print(f"  L1 error       : {seed_result['l1_error_mean']:.4f} ± {seed_result['l1_error_std']:.4f}")
    print(f"  Inference time : {seed_result['inference_time_ms_mean']:.1f} ± {seed_result['inference_time_ms_std']:.1f} ms")
    print(f"  Peak mem       : {seed_result['peak_mem_mib_mean']:.0f} MiB")
    print(f"  Avg power      : {avg_pwr_w:.1f} W")
    print(f"  Energy         : {total_kwh * 1000:.3f} Wh  |  CO2e: {co2e_g:.2f} g")
    print(f"  Wall-clock     : {wall_elapsed_s / 60:.1f} min  ({skipped} skipped)")

print("\nAll seeds complete.")

In [ ]:
# ── Cell 16: Aggregate across seeds ──────────────────────────────────────────
# §5.5 requires mean ± std across at least 3 runs.

def _agg(key):
    vals = [results_per_seed[str(s)][key] for s in SEEDS]
    return {"mean": float(np.mean(vals)), "std": float(np.std(vals)), "n_runs": len(SEEDS), "values": vals}

aggregate = {
    "l1_error":            _agg("l1_error_mean"),
    "inference_time_ms":   _agg("inference_time_ms_mean"),
    "peak_mem_mib":        _agg("peak_mem_mib_mean"),
    "avg_power_w":         _agg("avg_power_w"),
    "total_kwh":           _agg("total_kwh"),
    "co2e_g":              _agg("co2e_g"),
    "wall_time_s":         _agg("wall_time_s"),
}

print("Aggregate across 3 seeds")
print(f"  L1 error       : {aggregate['l1_error']['mean']:.4f} ± {aggregate['l1_error']['std']:.4f}")
print(f"  Inference time : {aggregate['inference_time_ms']['mean']:.1f} ± {aggregate['inference_time_ms']['std']:.1f} ms")
print(f"  Peak mem       : {aggregate['peak_mem_mib']['mean']:.0f} MiB")
print(f"  Total energy   : {aggregate['total_kwh']['mean'] * 1000:.3f} Wh/run")

In [ ]:
# ── Cell 17: Save results/baseline_metrics.json ───────────────────────────────

output = {
    "phase": 1,
    "model": MODEL_ID,
    "model_variant": "INT8_baseline",
    "unnorm_key": UNNORM_KEY,
    "seeds": SEEDS,
    "n_eval_episodes_per_run": N_EVAL_EPISODES,
    "hardware": hardware_profile,
    "per_seed": results_per_seed,
    "aggregate": aggregate,
}

out_path = RESULTS_DIR / "baseline_metrics.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Results saved → {out_path}")

# ── Verification checklist (§ plan Phase 1 Verification) ─────────────────────
print("\n── Verification ─────────────────────────────────────────────")

n_params_check = output["hardware"]["n_params_total"]
ok_params = 6e9 < n_params_check < 9e9
print(f"[{'OK' if ok_params else 'FAIL'}] Param count ~7.5B : {n_params_check/1e9:.3f}B")

peak_mem = aggregate["peak_mem_mib"]["mean"]
vram_limit_mib = hardware_profile["gpu_vram_gb"] * 1024
ok_mem = peak_mem < vram_limit_mib
print(f"[{'OK' if ok_mem else 'FAIL'}] Peak mem < VRAM   : {peak_mem:.0f} MiB < {vram_limit_mib:.0f} MiB")

required_keys = ["l1_error", "inference_time_ms", "peak_mem_mib", "avg_power_w", "total_kwh"]
ok_keys = all(k in aggregate for k in required_keys)
print(f"[{'OK' if ok_keys else 'FAIL'}] All 5 metric types present")

ok_runs = all(aggregate[k]["n_runs"] >= 3 for k in required_keys)
print(f"[{'OK' if ok_runs else 'FAIL'}] n_runs >= 3 for all metrics")

print("────────────────────────────────────────────────────────────")

In [ ]:
# ── Cell 18: Retrieve artifacts ───────────────────────────────────────────────
if IS_KAGGLE:
    # Kaggle auto-preserves /kaggle/working/ — no download step needed.
    # Pull output via: kaggle kernels output benjaminbrumm/vw-phase1-baseline -p <dir>
    import shutil, os

    # Copy hardware_profile.yaml into /kaggle/working/ for easy retrieval
    hw_src = Path("configs/hardware_profile.yaml")
    hw_dst = Path("/kaggle/working/hardware_profile.yaml")
    if hw_src.exists():
        shutil.copy(hw_src, hw_dst)
        print(f"Copied: {hw_src} → {hw_dst}")

    metrics_path = RESULTS_DIR / "baseline_metrics.json"
    print(f"\nArtifacts in /kaggle/working/:")
    for p in sorted(Path("/kaggle/working").rglob("*.json")) + sorted(Path("/kaggle/working").rglob("*.yaml")):
        print(f"  {p}  ({p.stat().st_size:,} bytes)")
else:
    from google.colab import files
    files.download(str(RESULTS_DIR / "baseline_metrics.json"))
    files.download("configs/hardware_profile.yaml")
    print("Files downloaded. Commit them:")
    print("  git add results/baseline_metrics.json configs/hardware_profile.yaml")
    print("  git commit -m '[phase1] add INT8 baseline metrics and hardware profile'")

## Results Summary

After running all cells, `results/baseline_metrics.json` contains the following
structure (values filled in by the evaluation run):

```
aggregate:
  l1_error          mean ± std  (n_runs=3)   ← §5.5 Compression vs. Accuracy baseline
  inference_time_ms mean ± std  (n_runs=3)   ← §5.5 Efficiency Gain baseline
  peak_mem_mib      mean ± std  (n_runs=3)   ← §5.5 Latency on Reference Profile
  avg_power_w       mean ± std  (n_runs=3)   ← §4.2 Energy baseline
  total_kwh         mean ± std  (n_runs=3)   ← §4.2 Energy baseline
```

These numbers are the **denominator** for Phase 3's efficiency gain and compression
vs. accuracy calculations.  Phase 2 (TN compression) and Phase 3 (evaluation)
read this file as their reference point.

### Challenge sections satisfied
- **§5.4** Accepted baseline (INT8 bitsandbytes) established
- **§5.5** Compression vs. Accuracy baseline: `l1_error` mean ± std
- **§5.5** Efficiency Gain baseline: `inference_time_ms` mean ± std
- **§5.5** Latency on Reference Profile: `inference_time_ms` mean on stated GPU
- **§5.2** Quantitative baseline for the 3-run mean ± std requirement
- **§6**   Hardware and energy data captured in `hardware_profile.yaml`